# SQL Cheat Sheet

SQL (Structured Query Language) is how you create tables in a relational database, put data in them, and ask questions of that data. The cards below cover the statements used most in data work: defining tables, querying and filtering rows, summarizing groups, combining tables, and changing data. Every example runs against the small sample tables in the next section, and the text under each example says what it returns.

Keywords are written in capitals by convention; SQL does not require it. Text values go in single quotes (`'ML'`), and every statement ends with a semicolon.

## Sample Tables

<style>
.sq-sheet{display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:14px;margin:1em 0;font-family:system-ui,sans-serif}
.sq-card{background:#fff;color:#0f172a;border:1px solid #cbd5e1;border-radius:12px;padding:14px 16px;box-shadow:0 1px 3px rgba(15,23,42,.12)}
.sq-head{display:flex;justify-content:space-between;align-items:center;gap:8px;margin-bottom:8px}
.sq-title{font-weight:700;font-size:1.05em;letter-spacing:.03em;text-transform:uppercase}
.sq-icon{background:#2e1065;color:#ddd6fe;border-radius:6px;min-width:2.1em;padding:.25em .5em;text-align:center;font:600 .85em ui-monospace,monospace;white-space:nowrap}
.sq-card pre{margin:0 0 8px;padding:8px 10px;background:#0f172a;color:#e2e8f0;border-radius:6px;font-size:.95em;white-space:pre-wrap;overflow-wrap:anywhere}
.sq-card p{margin:0;font-size:.95em;color:#334155}
.sq-card code{background:#e2e8f0;color:#0f172a;padding:0 .25em;border-radius:3px}
.sq-data{border-collapse:collapse;font-size:.9em;margin:.5em 0 1em}
.sq-data th,.sq-data td{border:1px solid #cbd5e1;padding:3px 10px;text-align:left}
.sq-data th{background:#ede9fe;color:#2e1065}
.sq-data td.num{text-align:right}
html[data-theme="dark"] .sq-card{background:#1e293b;color:#e2e8f0;border-color:#334155}
html[data-theme="dark"] .sq-card p{color:#cbd5e1}
html[data-theme="dark"] .sq-card code{background:#334155;color:#e2e8f0}
html[data-theme="dark"] .sq-data th,html[data-theme="dark"] .sq-data td{border-color:#475569}
html[data-theme="dark"] .sq-data th{background:#2e1065;color:#ede9fe}
</style>

The examples use three tables. `employees` lists current staff, `departments` lists departments and where they are, and `former_employees` lists people who have left. Evan works in Sales, which has no row in `departments`, and Research has no employees: those two gaps show how the joins differ.

<p><strong>employees</strong></p>
<table class="sq-data"><thead><tr><th>id</th><th>name</th><th>department</th><th>salary</th><th>hire_date</th></tr></thead><tbody><tr><td class="num">1</td><td>Alice</td><td>Data Science</td><td class="num">90000</td><td>2021-03-15</td></tr><tr><td class="num">2</td><td>Bob</td><td>ML</td><td class="num">85000</td><td>2022-07-01</td></tr><tr><td class="num">3</td><td>Charlie</td><td>Data Science</td><td class="num">80000</td><td>2023-01-10</td></tr><tr><td class="num">4</td><td>Diana</td><td>AI</td><td class="num">70000</td><td>2024-05-20</td></tr><tr><td class="num">5</td><td>Evan</td><td>Sales</td><td class="num">60000</td><td>2020-11-02</td></tr></tbody></table>

<p><strong>departments</strong></p>
<table class="sq-data"><thead><tr><th>name</th><th>location</th></tr></thead><tbody><tr><td>Data Science</td><td>Austin</td></tr><tr><td>ML</td><td>Boston</td></tr><tr><td>AI</td><td>Denver</td></tr><tr><td>Research</td><td>Seattle</td></tr></tbody></table>

<p><strong>former_employees</strong></p>
<table class="sq-data"><thead><tr><th>id</th><th>name</th></tr></thead><tbody><tr><td class="num">2</td><td>Bob</td></tr><tr><td class="num">7</td><td>Frank</td></tr></tbody></table>

Bob left and was later rehired, so he appears in both `employees` and `former_employees`.

## Define Tables

Data Definition Language (DDL) statements create and change the structure of tables, not the rows in them.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Create Table</span><span class="sq-icon" aria-hidden="true">+</span></div><pre>CREATE TABLE employees (
    id         INT PRIMARY KEY,
    name       VARCHAR(50),
    department VARCHAR(50),
    salary     DECIMAL(10, 2),
    hire_date  DATE
);</pre><p>Create a table and name each column with its data type. <code>PRIMARY KEY</code> makes <code>id</code> unique for every row.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Alter Table</span><span class="sq-icon" aria-hidden="true">~</span></div><pre>ALTER TABLE employees
ADD COLUMN email VARCHAR(100);</pre><p>Change a table's structure; here, add an <code>email</code> column. Existing rows get <code>NULL</code> in it.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Create View</span><span class="sq-icon" aria-hidden="true">[ ]</span></div><pre>CREATE VIEW high_earners AS
SELECT name, department, salary
FROM employees
WHERE salary &gt; 80000;</pre><p>Save a query under a name. <code>SELECT * FROM high_earners;</code> then returns Alice and Bob. A view stores the query, not a copy of the rows.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Truncate Table</span><span class="sq-icon" aria-hidden="true">∅</span></div><pre>TRUNCATE TABLE employees;</pre><p>Delete every row but keep the table and its columns. SQLite has no <code>TRUNCATE</code>; use <code>DELETE FROM employees;</code>.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Drop Table</span><span class="sq-icon" aria-hidden="true">✕</span></div><pre>DROP TABLE employees;</pre><p>Delete the table, its structure, and all of its rows. There is no undo.</p></div>
</div>

## Query Rows

A `SELECT` statement reads rows and never changes the data. Statements that read are sometimes called DQL (Data Query Language), an informal name rather than a category in the SQL standard.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Select Columns</span><span class="sq-icon" aria-hidden="true">&gt;_</span></div><pre>SELECT name, department
FROM employees;</pre><p>Return only the listed columns, for all five rows.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Select All Columns</span><span class="sq-icon" aria-hidden="true">*</span></div><pre>SELECT *
FROM employees;</pre><p>Return every column. Handy for a quick look; name the columns in code you keep.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Distinct Values</span><span class="sq-icon" aria-hidden="true">1</span></div><pre>SELECT DISTINCT department
FROM employees;</pre><p>Return each value once: Data Science, ML, AI, and Sales.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Sort Results</span><span class="sq-icon" aria-hidden="true">↓</span></div><pre>SELECT name, salary
FROM employees
ORDER BY salary DESC;</pre><p>Sort from highest to lowest salary. <code>ASC</code> (the default) sorts lowest first.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Limit Results</span><span class="sq-icon" aria-hidden="true">3</span></div><pre>SELECT name, salary
FROM employees
ORDER BY salary DESC
LIMIT 3;</pre><p>Return the first three rows: the top three earners, Alice, Bob, and Charlie. Without <code>ORDER BY</code>, which rows come first is not guaranteed.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Rename with AS</span><span class="sq-icon" aria-hidden="true">as</span></div><pre>SELECT name,
       salary / 12 AS monthly_pay
FROM employees;</pre><p>Compute a value per row and give the result column a name (an alias).</p></div>
</div>

## Filter Rows

`WHERE` keeps only the rows for which its condition is true.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Comparison</span><span class="sq-icon" aria-hidden="true">&gt;</span></div><pre>SELECT * FROM employees
WHERE salary &gt; 80000;</pre><p>Alice and Bob. Other operators: <code>=</code>, <code>&lt;&gt;</code> (not equal), <code>&lt;</code>, <code>&lt;=</code>, <code>&gt;=</code>.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Multiple Conditions</span><span class="sq-icon" aria-hidden="true">&amp;</span></div><pre>SELECT * FROM employees
WHERE department = &#x27;Data Science&#x27;
  AND salary &gt; 80000;</pre><p>Alice only. <code>AND</code> needs both conditions, <code>OR</code> needs either, and <code>NOT</code> reverses one. Use parentheses when you mix <code>AND</code> and <code>OR</code>.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">IN</span><span class="sq-icon" aria-hidden="true">∈</span></div><pre>SELECT * FROM employees
WHERE department IN (&#x27;Data Science&#x27;, &#x27;ML&#x27;, &#x27;AI&#x27;);</pre><p>Match any value in the list: everyone except Evan. A shorter way to write several <code>OR</code> conditions.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">BETWEEN</span><span class="sq-icon" aria-hidden="true">↔</span></div><pre>SELECT * FROM employees
WHERE salary BETWEEN 60000 AND 85000;</pre><p>Bob, Charlie, Diana, and Evan. <code>BETWEEN</code> includes both ends, so 60000 and 85000 match.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">LIKE</span><span class="sq-icon" aria-hidden="true">%</span></div><pre>SELECT * FROM employees
WHERE name LIKE &#x27;A%&#x27;;</pre><p>Names that start with A: Alice. <code>%</code> matches any number of characters and <code>_</code> exactly one, so <code>'_o%'</code> matches Bob.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">IS NULL</span><span class="sq-icon" aria-hidden="true">?</span></div><pre>SELECT * FROM employees
WHERE email IS NULL;</pre><p>Rows with no email yet (the column added in Alter Table): all five. <code>NULL</code> means a missing value. Test it with <code>IS NULL</code> or <code>IS NOT NULL</code>; <code>= NULL</code> is never true.</p></div>
</div>

## Summarize and Group

Aggregate functions turn many rows into one value. `GROUP BY` computes them once per group instead of once for the whole table.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Count Rows</span><span class="sq-icon" aria-hidden="true">#</span></div><pre>SELECT COUNT(*)
FROM employees;</pre><p>5. <code>COUNT(column)</code> counts only rows where that column is not <code>NULL</code>.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Average, Max, Min, Sum</span><span class="sq-icon" aria-hidden="true">Σ</span></div><pre>SELECT AVG(salary), MAX(salary),
       MIN(salary), SUM(salary)
FROM employees;</pre><p>77000, 90000, 60000, and 385000. All four skip <code>NULL</code> values.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Group By</span><span class="sq-icon" aria-hidden="true">▤</span></div><pre>SELECT department, COUNT(*),
       AVG(salary)
FROM employees
GROUP BY department;</pre><p>One row per department: Data Science 2 and 85000, ML 1 and 85000, AI 1 and 70000, Sales 1 and 60000. List more columns after <code>GROUP BY</code> to get one row per combination.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Having</span><span class="sq-icon" aria-hidden="true">▽</span></div><pre>SELECT department, AVG(salary)
FROM employees
GROUP BY department
HAVING AVG(salary) &gt; 80000;</pre><p>Data Science and ML. <code>WHERE</code> filters rows before grouping; <code>HAVING</code> filters groups after, so it can test an aggregate.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Count Unique Values</span><span class="sq-icon" aria-hidden="true">1</span></div><pre>SELECT COUNT(DISTINCT department)
FROM employees;</pre><p>4: count each department once.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Case</span><span class="sq-icon" aria-hidden="true">if</span></div><pre>SELECT name,
  CASE WHEN salary &gt;= 85000 THEN &#x27;High&#x27;
       WHEN salary &gt;= 70000 THEN &#x27;Mid&#x27;
       ELSE &#x27;Entry&#x27;
  END AS band
FROM employees;</pre><p>Label each row by the first condition it meets: Alice and Bob High, Charlie and Diana Mid, Evan Entry.</p></div>
</div>

## Join Tables

A join puts columns from two tables side by side, pairing rows where the `ON` condition is true. Here `e` and `d` are short aliases for the two tables.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Inner Join</span><span class="sq-icon" aria-hidden="true">⋂</span></div><pre>SELECT e.name, d.location
FROM employees e
INNER JOIN departments d
  ON e.department = d.name;</pre><p>Only rows that match in both tables: Alice, Bob, Charlie, and Diana. Evan (Sales) and Research drop out. <code>JOIN</code> alone means <code>INNER JOIN</code>.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Left Join</span><span class="sq-icon" aria-hidden="true">⟕</span></div><pre>SELECT e.name, d.location
FROM employees e
LEFT JOIN departments d
  ON e.department = d.name;</pre><p>Every employee, plus the department when one matches. Evan appears with <code>NULL</code> for <code>location</code>.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Right Join</span><span class="sq-icon" aria-hidden="true">⟖</span></div><pre>SELECT e.name, d.name, d.location
FROM employees e
RIGHT JOIN departments d
  ON e.department = d.name;</pre><p>Every department, plus its employees. Research appears with <code>NULL</code> for <code>e.name</code>. The same as a left join with the tables swapped.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Full Join</span><span class="sq-icon" aria-hidden="true">⟗</span></div><pre>SELECT e.name, d.name, d.location
FROM employees e
FULL JOIN departments d
  ON e.department = d.name;</pre><p>Every row from both tables: the four matches, Evan without a department, and Research without employees.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Find Unmatched Rows</span><span class="sq-icon" aria-hidden="true">∅</span></div><pre>SELECT e.name
FROM employees e
LEFT JOIN departments d
  ON e.department = d.name
WHERE d.name IS NULL;</pre><p>Employees whose department is missing from <code>departments</code>: Evan. A common check for data that does not line up.</p></div>
</div>

## Combine Query Results

Set operations stack the rows of two queries. Both queries must return the same number of columns, with compatible types.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Union</span><span class="sq-icon" aria-hidden="true">∪</span></div><pre>SELECT name FROM employees
UNION
SELECT name FROM former_employees;</pre><p>All names, with duplicates removed: six names, with Bob once.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Union All</span><span class="sq-icon" aria-hidden="true">∪+</span></div><pre>SELECT name FROM employees
UNION ALL
SELECT name FROM former_employees;</pre><p>All seven rows, keeping duplicates; Bob appears twice. Faster than <code>UNION</code> because it does not check for duplicates.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Intersect</span><span class="sq-icon" aria-hidden="true">∩</span></div><pre>SELECT name FROM employees
INTERSECT
SELECT name FROM former_employees;</pre><p>Names in both results: Bob.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Except</span><span class="sq-icon" aria-hidden="true">−</span></div><pre>SELECT name FROM employees
EXCEPT
SELECT name FROM former_employees;</pre><p>Names in the first result but not the second: Alice, Charlie, Diana, and Evan. Oracle calls it <code>MINUS</code>.</p></div>
</div>

## Subqueries and CTEs

A query can use the result of another query. A subquery sits inside the statement; a common table expression (CTE) names it first with `WITH`, which makes longer queries easier to read.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Subquery</span><span class="sq-icon" aria-hidden="true">( )</span></div><pre>SELECT name, salary
FROM employees
WHERE salary &gt; (
    SELECT AVG(salary) FROM employees
);</pre><p>Employees paid more than the company average of 77000: Alice, Bob, and Charlie. The inner query runs first and returns one value.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Common Table Expression</span><span class="sq-icon" aria-hidden="true">with</span></div><pre>WITH dept_avg AS (
    SELECT department,
           AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department
)
SELECT e.name, e.salary, d.avg_salary
FROM employees e
JOIN dept_avg d
  ON e.department = d.department
WHERE e.salary &gt; d.avg_salary;</pre><p>Employees paid more than their own department's average: Alice (90000 against 85000). <code>dept_avg</code> exists only for this statement.</p></div>
</div>

## Change Data

Data Manipulation Language (DML) statements add, change, and remove rows. Run the matching `SELECT` first to see which rows a `WHERE` condition selects.

<div class="sq-sheet" role="list">
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Insert Rows</span><span class="sq-icon" aria-hidden="true">+</span></div><pre>INSERT INTO employees
    (id, name, department, salary, hire_date)
VALUES (6, &#x27;Fay&#x27;, &#x27;AI&#x27;, 75000, &#x27;2025-01-06&#x27;);</pre><p>Add a row. List the columns so the statement still works if the table gains columns later.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Update Rows</span><span class="sq-icon" aria-hidden="true">✎</span></div><pre>UPDATE employees
SET salary = 95000
WHERE id = 1;</pre><p>Change Alice's salary. Without <code>WHERE</code>, <code>UPDATE</code> changes every row.</p></div>
<div class="sq-card" role="listitem"><div class="sq-head"><span class="sq-title">Delete Rows</span><span class="sq-icon" aria-hidden="true">✕</span></div><pre>DELETE FROM employees
WHERE id = 6;</pre><p>Remove Fay's row. Without <code>WHERE</code>, <code>DELETE</code> removes every row.</p></div>
<!-- slot: more_cards --><!-- /slot: more_cards -->
</div>

## How a Query Runs

The clauses of a `SELECT` are written in one order and evaluated in another:

1. `FROM` and `JOIN` choose the tables and pair their rows.
2. `WHERE` keeps the matching rows.
3. `GROUP BY` forms groups, and `HAVING` keeps the matching groups.
4. `SELECT` computes the output columns, and `DISTINCT` removes repeated rows.
5. `ORDER BY` sorts the result, and `LIMIT` keeps the first rows.

This order explains common errors: `WHERE` cannot use an aggregate such as `AVG(salary)` because groups do not exist yet (use `HAVING`), and in most databases `WHERE` cannot use a column alias defined in `SELECT`.

## Differences Between Databases

The examples follow standard SQL and run as written in PostgreSQL and in SQLite 3.39 or later. Other databases differ in a few places:

- **Row limits:** SQL Server writes `SELECT TOP 3 ...`, and Oracle writes `FETCH FIRST 3 ROWS ONLY` after `ORDER BY`.
- **Joins:** MySQL has no `FULL JOIN`; combine a left join and a right join with `UNION`. SQLite before 3.39 has no `RIGHT JOIN` or `FULL JOIN`.
- **Set operations:** MySQL supports `INTERSECT` and `EXCEPT` from version 8.0.31.
- **Truncate:** SQLite has no `TRUNCATE`; use `DELETE FROM` without `WHERE`.

<!-- slot: book_notes --><!-- /slot: book_notes -->